## EDA on label from User

QUESTION 1: Find all possible unique values
- Category
- Subcategory
- Root_code
- Sub_code

In [ ]:
import pandas as pd

In [ ]:

df = pd.read_excel('label.xlsx')

print("--- 1. UNIQUE VALUES IN DATASET ---")
print(f"Categories ({df['Category'].nunique()}): {df['Category'].dropna().unique()}")
print(f"Subcategories ({df['Subcategory'].nunique()}): {df['Subcategory'].dropna().unique()}")
print(f"Root Codes ({df['root_code'].nunique()}): {df['root_code'].dropna().unique()}")
print(f"Sub Codes ({df['sub_code'].nunique()}): {df['sub_code'].dropna().unique()}\n")

QUESTION 2: Find mappings and human errors

In [ ]:
print("--- 2. MAPPING RULES & ANOMALY DETECTION ---")

# Group by the human text labels and see what codes they mapped to, counting occurrences
mapping_df = df.groupby(
    ['Category', 'Subcategory', 'root_code', 'sub_code'], 
    dropna=False
).size().reset_index(name='Count')

# Sort so that for each Category/Subcategory, the most frequent mapping is at the top
mapping_df = mapping_df.sort_values(
    by=['Category', 'Subcategory', 'Count'], 
    ascending=[True, True, False]
)

# Calculate the total times a Category/Subcategory pair appears
mapping_df['Total_Category_Occurrences'] = mapping_df.groupby(['Category', 'Subcategory'])['Count'].transform('sum')

# Calculate the percentage of times THIS specific mapping was used
mapping_df['Mapping_Confidence_%'] = (mapping_df['Count'] / mapping_df['Total_Category_Occurrences']) * 100

print("\nAll Mappings (Sorted by frequency):")
display(mapping_df) # Use print(mapping_df) if not in Jupyter

In [ ]:
# --- ISOLATING THE ERRORS ---
# Assuming that whatever mapping happens the majority of the time (>50%) is the "Correct" rule...
# Anything else is likely a human annotation error.

print("\n--- LIKELY HUMAN ANNOTATION ERRORS ---")
# Filter for mappings that happen less than 20% of the time for that specific category
anomalies = mapping_df[mapping_df['Mapping_Confidence_%'] < 20.0]

if anomalies.empty:
    print("Good news! No major anomalies found. The mapping is consistent.")
else:
    print("Found potential mislabels! Annotators likely messed these up:")
    display(anomalies)

In [ ]:
# ==========================================
# CELL 1: EYEBALL SPECIFIC CATEGORIES
# ==========================================
from IPython.display import Image, display
import os

# 1. Define the specific pair you want to investigate
TARGET_CATEGORY = "Insert Category Here"       # e.g., "Medical Document"
TARGET_SUBCAT = "Insert Subcategory Here"      # e.g., "Lab Results"

# 2. Filter the existing 'df' for this specific pair
subset_df = df[(df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)]

print(f"Found {len(subset_df)} documents for {TARGET_CATEGORY} -> {TARGET_SUBCAT}\n")

# 3. Display the images (Showing the first 10 to avoid crashing the notebook)
for idx, row in subset_df.head(10).iterrows():
    print("-" * 50)
    print(f"Row Index: {idx}")
    print(f"Current Labels -> root_code: [{row['root_code']}] | sub_code: [{row['sub_code']}]")
    print(f"Filepath: {row['Filepath']}")
    
    filepath = str(row['Filepath'])
    if os.path.exists(filepath):
        # Display the image directly in the notebook output
        display(Image(filename=filepath, width=600))
    else:
        print(f"Image not found at path: {filepath}")

In [ ]:
# ==========================================
# CELL 2: BATCH REPLACE & SAVE
# ==========================================

# 1. Define the pair you want to fix
TARGET_CATEGORY = "Insert Category Here"
TARGET_SUBCAT = "Insert Subcategory Here"

# 2. Define the CORRECT codes they should be mapped to
CORRECT_ROOT_CODE = "NON"
CORRECT_SUB_CODE = "OTH"

# 3. Create the filter condition
condition = (df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)

# 4. Use .loc to safely update the values in the original dataframe
df.loc[condition, 'root_code'] = CORRECT_ROOT_CODE
df.loc[condition, 'sub_code'] = CORRECT_SUB_CODE

print(f"Successfully updated {condition.sum()} rows to {CORRECT_ROOT_CODE} -> {CORRECT_SUB_CODE}.")

# 5. Export the cleaned data to CSV
output_filename = 'label_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"Cleaned dataset exported as: {output_filename}")

## Evaluation

In [ ]:
 
import math
import pandas as pd
from typing import Literal, Optional
from pydantic import BaseModel
from openai import AzureOpenAI, BadRequestError
 
# ==========================================
# 1. DEFINE STRUCTURED OUTPUT SCHEMAS
# ==========================================
 
class TriageOutput(BaseModel):
    chain_of_thought: str
    routing_decision: Literal["medical", "processable_non_medical", "trash_others"]
 
class NonMedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "financial_bankstatement", "financial_bookbank",
        "financial_companyregistration", "financial_others",
        "id_driverlicense", "id_fatca_w9", "id_foreignerconfirmationform",
        "id_foreigner_nationalid", "id_visastamp", "id_passport",
        "id_statelessid", "id_thainationalid",
        "id_workpermit", "id_others"
        # NOTE: id_fullthaivisa + id_passportstamp merged into id_visastamp -- these two
        # almost always appear on the same physical page, so splitting them was creating
        # noisy boundary cases with no real downstream benefit.
    ]
 
class MedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "medical_clinical", "medical_healthcheck",
        "medical_lab", "medical_others"
    ]
 
# ==========================================
# 2. DEFINE PROMPT VARIABLES
# ==========================================
 
# --- AGENT 1: TRIAGE PROMPTS ---
# CHANGED (v4): same decision logic as v3, rewritten for token efficiency -- collapsed
# repeated warnings, pulled the eForm test out as one shared rule instead of restating it
# under both 'processable_non_medical' and 'trash_others', shorter sentences throughout.
# ~35% fewer tokens than v3 with no loss of the watermark / eForm / OCR-noise / partial-
# legibility fixes.
AGENT1_SYS_PROMPT = """You are a Triage Routing Agent. Classify OCR text (sometimes markdown with headers/tables from a Document Intelligence engine) into exactly one bucket: medical, processable_non_medical, or trash_others. Use markdown structure (tables, headers) as a signal when present. Do not process PII.
 
### Ignore this watermark
An insurance disclaimer resembling  (+ a date/time stamp) appears on many unrelated document types. It is never evidence of document type -- ignore it completely when deciding.
 
### 1. medical
Contains real patient healthcare data (clinical notes, lab metrics, health-check parameters, prescriptions).
- Hospital name/logo alone is not medical. Billing/invoices/receipts -> trash_others.
- Garbled OCR from blurry scans that still shows technical-looking terms next to numbers/units/ranges, or positive/negative or normal/abnormal flags -> treat as low-confidence medical, not trash. Flag the uncertainty in chain_of_thought.
 
### 2. processable_non_medical
Route here if ANY native-structure anchor is present, even without certainty on the exact subtype (that's the specialist's job):
- ID card's own layout: issuing country name/code (e.g. "LAO PDR"), a name+DOB block, or a photo placeholder -- as the document's own structure.
- Passport/visa/immigration stamp: "DEPARTED"/"ADMITTED"/"ENTRY" near date-like text. Stamp text is curved and often OCR-garbled -- messy digits near these keywords still count.
- Bank/account identity: account number + bank name/branch, a transaction ledger (especially as a markdown table), or mobile banking markers.
- Corporate registration number + certification/signature block.
- Partial legibility: if most text is unreadable (e.g. Lao script) but legible fragments show any anchor above, still route here.
- When torn, prefer this bucket -- a specialist landing it in an "_others" leaf is a safe, expected outcome, not an error.
 
**eForm test:** an ID number, name, DOB, or premium/sum-insured figure that is a FILLED-IN FIELD inside an insurance application template is not an anchor -- it's applicant metadata, not a native document. Only count anchors from the document's own structure (a card, a passport/visa page, a bank ledger, a registration certificate) -- never from a "label: value" application form.
 
### 3. trash_others
Default when no anchor above is present. Always trash regardless of anchors:
- Hospital billing/receipts.
- Insurance application eForms/premium declarations -- "label: value" templates with applicant/agent/product fields, ทุนประกัน/เบี้ยประกัน, and the watermark -- even if they contain an embedded ID number or money figure (see eForm test).
- Portrait photos/ID placeholders with no issuing text.
- Anything else lacking every anchor above.
 
In chain_of_thought: name the anchor found (or state none found), and note explicitly if you're disregarding an insurance watermark that would otherwise bias toward trash_others."""
 
AGENT1_USER_PROMPT = "Evaluate and triage the following raw OCR text:\n{ocr_text}"
 
 
# --- AGENT 2: NON-MED SPECIALIST PROMPTS ---
AGENT2_SYS_PROMPT = """You are an expert Non-Medical Document Specialist. You receive text (often markdown-formatted with layout tags from a Document Intelligence engine) that has already passed an initial triage anchor-check for being a Financial or Identification document -- it may still turn out to be a borderline or imperfect match, or occasionally a mis-routed insurance application eForm. Your task is to perform granular classification.
 
Analyze the layout clues, boilerplate language, and structural anchors. Use markdown structure (tables, headers) where present. Do not look for or process private personal data.
 
### Defense-in-depth: catch mis-routed insurance eForms
If you find the document is actually a system-printed insurance application template -- "label: value" fields for applicant/agent/product/plan, "ทุนประกัน" (sum insured), "เบี้ยประกัน" (premium), plus an insurance watermark -- rather than a document with its own native ID-card/passport/bank/registration structure, treat it as NOT genuinely financial or ID, even though triage passed it through. Pick the closest "_others" leaf (financial_others or id_others) and note your uncertainty in chain_of_thought so it surfaces for human review, rather than confidently forcing it into a specific subtype like id_thainationalid.
 
### Subclass Evaluation Guides:
- financial_bankstatement: Look for running transactional ledgers, money transfers, account balances, and mobile banking app markers -- ideally a markdown table of dated transaction rows.
- financial_bookbank: Look for static account identity headers (bilingual accounts, branches, bank names) without long subsequent lists of transfer ledgers.
- financial_companyregistration: Look for formal commercial corporate registry verbiage and certification signatures.
- financial_others: Use this if the document is clearly financial in nature but doesn't cleanly match bankstatement, bookbank, or companyregistration -- or if you suspect it's actually an insurance eForm that leaked through triage (see above).
- id_fatca_w9: Look for specific US tax withholding terminology, backup withholding rules, and entity declaration sections.
- id_passport: Look for the passport bio data page -- photo/data block, passport number, nationality, MRZ-style lines.
- id_visastamp: Look for a Thai visa page and/or an immigration control stamp together -- visa category codes, "DEPARTED"/"ADMITTED"/"ENTRY" keywords, and (often garbled, since stamp text is curved) date-like digit strings near them. This merges what used to be two separate classes (full Thai visa, passport stamp) since they almost always appear on the same physical page.
- id_workpermit: Look for labor authorization rules and official employment permission contexts.
- id_thainationalid / id_foreigner_nationalid: Look for national demographic card headers and identity issuing text blocks. Foreign-script documents (e.g. a Lao national ID) still count here even if large portions are unreadable by OCR -- look for legible fragments like the issuing country name/code, name, or a date-of-birth pattern.
- id_others: Use this if the document is clearly an official ID-type artifact but doesn't cleanly match any specific subtype above, OR for the borderline leaked-eForm case above. This is an expected, valid outcome for borderline documents -- don't force a document into a specific subtype it doesn't structurally match.
 
Provide your step-by-step structural rationale in 'chain_of_thought' before selecting the final leaf node subcategory."""
 
AGENT2_USER_PROMPT = "Perform deep classification on this verified processable text:\n{ocr_text}"
 
 
# --- AGENT 3: MEDICAL SPECIALIST PROMPTS ---
AGENT3_SYS_PROMPT = """You are an expert Medical Document Specialist. You receive text that has already been verified as an official medical or healthcare document. Your task is to perform granular classification.
 
Analyze clinical boilerplate language, diagnostic formats, and structural indicators. Do not attempt to analyze or extract confidential patient names or raw medical history.
 
### Subclass Evaluation Guides:
- medical_clinical: Look for outpatient/inpatient notes (OPD/IPD), physical exam details, hospital visit dates, physician signatures, and medication dosages/frequencies (e.g., "mg", "ครั้ง").
- medical_healthcheck: Look for standardized checkup checklists, physiological summary metrics (e.g., "min", "bpm"), and categorical evaluations of bodily systems (e.g., "normal", "abnormal", "ปกติ", "ไม่พบ").
- medical_lab: Look for quantitative lab panels, chemical compound analysis, test execution timestamps, measurement metrics (e.g., "mg/dL"), or diagnostic presence indicators (e.g., "positive", "negative").
- medical_others: Select this if the document is clearly medical but does not visually or structurally conform to an OPD/IPD sheet, a regular checkup report, or a quantitative lab metric page.
 
Provide your step-by-step structural rationale in 'chain_of_thought' before selecting the final leaf node subcategory."""
 
AGENT3_USER_PROMPT = "Perform deep clinical classification on this verified medical text:\n{ocr_text}"

# ==========================================
# 3. CORE COGNITIVE PIPELINE
# ==========================================

def calculate_confidence(response_logprobs) -> float:
    """Whole-response confidence (legacy). Kept for backward compatibility / comparison.
    NOTE: this dilutes the signal with free-text chain_of_thought tokens -- prefer
    calculate_field_confidence for anything you actually act on (e.g. review-queue thresholds)."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0
    token_logprobs = [t.logprob for t in response_logprobs.content if t.logprob is not None]
    if not token_logprobs:
        return 0.0
    avg_logprob = sum(token_logprobs) / len(token_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def calculate_field_confidence(response_logprobs, field_value: str) -> float:
    """Isolates and averages logprobs only for the tokens making up the classification
    enum value itself (e.g. 'processable_non_medical' or 'id_passport'), rather than the
    whole JSON payload including the free-text chain_of_thought. This is a much better
    proxy for "how sure was the model about the label" and is what should feed a
    human-review threshold.

    Falls back to calculate_confidence if the token span can't be located (e.g. due to
    tokenizer quirks), so it always returns a usable number.
    """
    if not response_logprobs or not response_logprobs.content:
        return 0.0

    tokens = response_logprobs.content
    token_strs = [t.token for t in tokens]

    # Reconstruct the running string and find which token indices correspond to
    # the field_value substring (the enum's own text), searching from the end since
    # 'routing_decision'/'subcategory' is emitted after chain_of_thought in our schemas.
    running = ""
    span_start, span_end = None, None
    for i, tok in enumerate(token_strs):
        prev_len = len(running)
        running += tok
        if field_value in running[max(0, prev_len - len(field_value)):]:
            end_idx = i
            # walk backward to find the first token index where field_value starts
            partial = ""
            start_idx = i
            for j in range(i, -1, -1):
                partial = token_strs[j] + partial
                start_idx = j
                if field_value in partial:
                    break
            span_start, span_end = start_idx, end_idx

    if span_start is None:
        return calculate_confidence(response_logprobs)

    span_logprobs = [tokens[i].logprob for i in range(span_start, span_end + 1) if tokens[i].logprob is not None]
    if not span_logprobs:
        return calculate_confidence(response_logprobs)

    avg_logprob = sum(span_logprobs) / len(span_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def run_cascade_pipeline(ocr_text: str, client: AzureOpenAI, model_name: str = "gpt-4o") -> dict:
    """Executes the three-stage cascading architecture on a single piece of text."""

    # ----------------------------------------------------
    # STAGE 1: Triage Routing
    # ----------------------------------------------------
    triage_response = client.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": AGENT1_SYS_PROMPT},
            {"role": "user", "content": AGENT1_USER_PROMPT.format(ocr_text=ocr_text)}
        ],
        response_format=TriageOutput,
        logprobs=True,
        top_logprobs=1
    )

    triage_data = triage_response.choices[0].message.parsed
    route = triage_data.routing_decision
    triage_conf = calculate_field_confidence(triage_response.choices[0].logprobs, route)

    # ----------------------------------------------------
    # STAGE 2A: Conditional Non-Med Processing
    # ----------------------------------------------------
    if route == "processable_non_medical":
        spec_response = client.beta.chat.completions.parse(
            model=model_name,
            messages=[
                {"role": "system", "content": AGENT2_SYS_PROMPT},
                {"role": "user", "content": AGENT2_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=NonMedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)

        return {
            "triage_decision": route,
            "triage_reason": triage_data.chain_of_thought,
            "triage_confidence": triage_conf,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf
        }

    # ----------------------------------------------------
    # STAGE 2B: Conditional Med Processing
    # ----------------------------------------------------
    elif route == "medical":
        spec_response = client.beta.chat.completions.parse(
            model=model_name,
            messages=[
                {"role": "system", "content": AGENT3_SYS_PROMPT},
                {"role": "user", "content": AGENT3_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=MedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)

        return {
            "triage_decision": route,
            "triage_reason": triage_data.chain_of_thought,
            "triage_confidence": triage_conf,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf
        }

    # If routed to Trash, exit early without calling any specialist agent
    return {
        "triage_decision": route,
        "triage_reason": triage_data.chain_of_thought,
        "triage_confidence": triage_conf,
        "final_subcategory": f"routed_to_{route}",
        "specialist_reason": "Skipped - Cleaned out as unstructured noise by triage agent.",
        "specialist_confidence": 100.0
    }

# ==========================================
# 4. DATAFRAME EVALUATION EXECUTION
# ==========================================

def evaluate_experiment(csv_path: str, endpoint: str, api_key: str):
    """Reads the evaluation data, executes the cascade pipeline, and returns results."""

    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-08-01-preview"
    )

    df = pd.read_csv(csv_path)
    experimental_results = []

    print(f"Starting complete cascade experiment on {len(df)} documents...")

    for idx, row in df.iterrows():
        ocr_text = str(row['ocr_text'])

        try:
            # Run entire multi-agent cascade pipeline
            res = run_cascade_pipeline(ocr_text, client)
        except BadRequestError as e:
            # Catch the specific Azure Content Filter / Jailbreak error
            error_message = str(e).lower()
            if "content management policy" in error_message or "jailbreak" in error_message:
                print(f"[WARNING] Doc {idx+1} blocked by Azure Jailbreak Filter. Skipping...")

                # Provide fallback data so the dataframe doesn't break
                res = {
                    "triage_decision": "blocked_by_firewall",
                    "triage_reason": "Azure OpenAI Content Filter flagged this OCR text as a jailbreak attempt.",
                    "triage_confidence": 0.0,
                    "final_subcategory": "error_content_filter",
                    "specialist_reason": "Skipped due to API security policy.",
                    "specialist_confidence": 0.0
                }
            else:
                # If it's a different BadRequest (like a missing parameter), crash normally so you can fix it
                raise e
        except Exception as e:
            # Catch any other random API timeouts or disconnects
            print(f"[ERROR] Doc {idx+1} failed due to unexpected error: {e}")
            res = {
                "triage_decision": "api_error",
                "triage_reason": str(e),
                "triage_confidence": 0.0,
                "final_subcategory": "api_error",
                "specialist_reason": "API disconnected or failed.",
                "specialist_confidence": 0.0
            }
        # Merge metrics back with the original ground-truth markers
        combined_row = {
            "input_category_label": row.get('category'),
            "input_subcategory_label": row.get('subcategory'),
            **res
        }
        experimental_results.append(combined_row)

        print(f"[{idx+1}/{len(df)}] {row['filename']} Triage Macro: {res['triage_decision']} -> Final Leaf Subclass: {res['final_subcategory']}")

    output_df = pd.DataFrame(experimental_results)
    output_df.to_csv("experiment_full_cascade_results.csv", index=False)
    print("\nFull evaluation pipeline complete. Results stored in 'experiment_full_cascade_results.csv'")
    return output_df

# To run in your notebook cell:
# df_results = evaluate_experiment("test_data.csv", "https://your-endpoint.openai.azure.com/", "your-api-key")